In [ ]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ No GPU detected")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U \
    bitsandbytes \
    transformers \
    peft \
    trl \
    accelerate \
    datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.0 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)

from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("All fine-tuning imports successful!")

PyTorch: 2.11.0+cu128
CUDA available: True
bitsandbytes: 0.50.2
All fine-tuning imports successful!


In [ ]:
import importlib.metadata

packages = [
    "bitsandbytes",
    "peft",
    "trl",
    "accelerate",
    "datasets",
    "transformers"
]

for package in packages:
    print(package, ":", importlib.metadata.version(package))

bitsandbytes : 0.50.2
peft : 0.20.0
trl : 1.13.0
accelerate : 1.15.0
datasets : 5.0.1
transformers : 5.17.0


In [ ]:
import torch
import bitsandbytes as bnb

print("GPU:", torch.cuda.get_device_name(0))

# Create a small GPU tensor
x = torch.randn(4, 4, device="cuda")

# Test a bitsandbytes 4-bit linear layer
layer = bnb.nn.Linear4bit(
    4,
    4,
    bias=False,
    compute_dtype=torch.float16,
    compress_statistics=True,
    quant_type="nf4"
).to("cuda")

# Initialize quantized weights
layer = layer.cuda()

output = layer(x)

print("bitsandbytes GPU test successful!")
print("Output shape:", output.shape)

GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


bitsandbytes GPU test successful!
Output shape: torch.Size([4, 4])


In [ ]:
from google.colab import userdata
from transformers import GemmaTokenizer

In [ ]:
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
model_id = "google/gemma-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=os.environ["HF_TOKEN"]
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=os.environ["HF_TOKEN"]
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
text= "Quote: Imagination is more,"
device="cuda:0"
inputs= tokenizer(text,return_tensors="pt").to(device)

outputs= model.generate(**inputs,max_new_tokens=30)
print(tokenizer.decode(outputs[0],skip_special_tokens=True))


Quote: Imagination is more, than knowledge.

I am a self-taught artist, born in 1985 in the beautiful city of Porto Alegre, Brazil.




In [ ]:
text= "Quote: Imagination is more,"
device="cuda:0"
inputs= tokenizer(text,return_tensors="pt").to(device)

outputs= model.generate(**inputs,max_new_tokens=30)
print(tokenizer.decode(outputs[0],skip_special_tokens=True))


Quote: Imagination is more, than knowledge.

I am a self-taught artist, born in 1985 in the beautiful city of Porto Alegre, Brazil.




In [ ]:
os.environ["WANDB_DISABLED"] = "false"

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,

    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    task_type="CAUSAL_LM",
)

In [ ]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")

data = data.map(
    lambda samples: tokenizer(samples["quote"]),
    batched=True
)

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
data['train']['quote']

Column(['“Be yourself; everyone else is already taken.”', "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”", '“So many books, so little time.”', '“A room without books is like a body without a soul.”', ...])

In [ ]:
data['train']

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 2508
})

In [ ]:
import transformers

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=data["train"],

    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=100,
        learning_rate=2e-4,

        fp16=False,
        bf16=False,

        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),

    peft_config=lora_config,
)

Building labels for train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
1,2.680069
2,1.701774
3,2.610207
4,2.845090
5,2.404137
6,2.518052
7,2.955647
8,2.286492
9,3.261146
10,2.315543


TrainOutput(global_step=100, training_loss=2.1305582690238953, metrics={'train_runtime': 171.4049, 'train_samples_per_second': 2.334, 'train_steps_per_second': 0.583, 'total_flos': 189744345784320.0, 'train_loss': 2.1305582690238953, 'epoch': 0.1594896331738437})

In [ ]:
trainer.save_model("gemma-2b-finetuned")

tokenizer.save_pretrained("gemma-2b-finetuned")

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!


In [ ]:
text= "Quote: I'm selfish,"
device="cuda:0"
inputs= tokenizer(text,return_tensors="pt").to(device)

outputs= model.generate(**inputs,max_new_tokens=30)
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Quote: I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me


In [ ]:
text= "Quote: Be who you are and say what you feel,"
device="cuda:0"
inputs= tokenizer(text,return_tensors="pt").to(device)

outputs= model.generate(**inputs,max_new_tokens=20)
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Quote: Be who you are and say what you feel, because those who mind don't matter and those who matter don't mind.

I'


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

source = "/content/gemma-2b-finetuned"

destination = "/content/drive/MyDrive/gemma-2b-finetuned"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Model saved successfully to Google Drive!")

Model saved successfully to Google Drive!


In [ ]:
import os

path = "/content/drive/MyDrive/gemma-2b-finetuned"

print(os.listdir(path))

['adapter_config.json', 'README.md', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors']


In [ ]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 26.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

# Reload the updated token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("Token reloaded!")
print("Token exists:", os.environ["HF_TOKEN"] is not None)
print("Token starts correctly:", os.environ["HF_TOKEN"].startswith("hf_"))
print("Token length:", len(os.environ["HF_TOKEN"]))

Token reloaded!
Token exists: True
Token starts correctly: True
Token length: 37


In [ ]:
from huggingface_hub import HfApi
import os

token = os.environ["HF_TOKEN"].strip()

api = HfApi(token=token)

user_info = api.whoami()

print("Authenticated successfully!")
print("Username:", user_info["name"])

Authenticated successfully!
Username: deepak2310


In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ["HF_TOKEN"])

api.upload_folder(
    folder_path="/content/gemma-2b-finetuned",
    repo_id="deepak2310/gemma-2b-english-quotes",
    repo_type="model"
)

print("Model uploaded successfully!")

Model uploaded successfully!
